# Multimodal AI Model Exploration

This notebook tests multiple AI modalities using Google Gemini and Hugging Face.

Modalities covered:

- Text → Text
- Text → Image
- Image → Text
- Audio → Text
- Text → Audio
- Text → Video
- Video → Text


## 1. Install Required Libraries


In [ ]:
!pip install -q google-genai huggingface_hub python-dotenv pillow ipython


## 2. Import Libraries and Load API Keys


In [ ]:
import os, time, wave
from dotenv import load_dotenv
from PIL import Image
from IPython.display import display, Audio
from google import genai
from google.genai import types
from huggingface_hub import InferenceClient

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
hf_client = InferenceClient(token=HF_TOKEN)

print('Gemini key loaded:', GEMINI_API_KEY is not None)
print('Hugging Face token loaded:', HF_TOKEN is not None)

def save_pcm_as_wav(filename, pcm_data, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, 'wb') as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm_data)

def wait_for_gemini_file_active(client, uploaded_file, max_wait_seconds=120):
    start_time = time.time()
    file_obj = client.files.get(name=uploaded_file.name)
    while getattr(file_obj, 'state', None) and file_obj.state.name == 'PROCESSING':
        if time.time() - start_time > max_wait_seconds:
            raise TimeoutError('Video file is still processing.')
        time.sleep(5)
        file_obj = client.files.get(name=uploaded_file.name)
    if getattr(file_obj, 'state', None) and file_obj.state.name != 'ACTIVE':
        raise RuntimeError(f'Gemini file is not active. Current state: {file_obj.state.name}')
    return file_obj


## 3. Text → Text using Gemini 2.5 Flash


In [ ]:
prompt = 'Explain multimodal AI in simple terms with one real-world example.'
response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt
)
print(response.text)


## 4. Text → Image using Stable Diffusion XL


In [ ]:
image_prompt = 'A futuristic classroom where students use multimodal AI tools, cinematic lighting'
generated_image = hf_client.text_to_image(
    image_prompt,
    model='stabilityai/stable-diffusion-xl-base-1.0'
)
display(generated_image)
generated_image.save('generated_image.png')


## 5. Image → Text using Gemini 2.5 Flash

Place an image named `sample_image.jpg` in this folder before running.


In [ ]:
image_path = 'sample_image.jpg'
img = Image.open(image_path)
display(img)
response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=['Describe this image in detail.', img]
)
print(response.text)


## 6. Audio → Text using Whisper Large v3 Turbo

Place an audio file named `sample_audio.mp3` in this folder before running.


In [ ]:
audio_path = 'sample_audio.mp3'
result = hf_client.automatic_speech_recognition(
    audio_path,
    model='openai/whisper-large-v3-turbo'
)
print(result.get('text', result) if isinstance(result, dict) else result)


## 7. Text → Audio using Gemini TTS


In [ ]:
tts_text = 'Hello, this is a text to speech example for my multimodal AI assignment.'
response = gemini_client.models.generate_content(
    model='gemini-2.5-flash-preview-tts',
    contents=tts_text,
    config=types.GenerateContentConfig(
        response_modalities=['AUDIO'],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name='Kore')
            )
        ),
    ),
)
audio_data = response.candidates[0].content.parts[0].inline_data.data
save_pcm_as_wav('generated_audio.wav', audio_data)
Audio('generated_audio.wav')


## 8. Text → Video using Text → Image → Video Pipeline

This creates a starter image from text, then converts that image to video.


In [ ]:
#video_prompt = 'A small robot walking through a futuristic classroom, cinematic lighting'
#starter_image = hf_client.text_to_image(
    #video_prompt,
    #model='stabilityai/stable-diffusion-xl-base-1.0'
#)
#starter_image.save('text_to_video_starter_image.png')
#display(starter_image)
#with open('text_to_video_starter_image.png', 'rb') as f:
    #video_bytes = hf_client.image_to_video(
        #f.read(),
        #model='Lightricks/LTX-Video-0.9.8-13B-distilled'
    #)
#with open('generated_video.mp4', 'wb') as f:
    #f.write(video_bytes)
#print('Saved generated_video.mp4')


In [ ]:
from huggingface_hub import InferenceClient
from IPython.display import Video

video_prompt = "A small robot walking through a futuristic classroom, cinematic lighting"

# Direct text-to-video client using Hugging Face Inference Provider
video_client = InferenceClient(
    provider="fal-ai",
    api_key=HF_TOKEN,
)

video = video_client.text_to_video(
    video_prompt,
    model="Wan-AI/Wan2.2-TI2V-5B",
)

with open("generated_video.mp4", "wb") as f:
    f.write(video)

print("Saved generated_video.mp4")

Video("generated_video.mp4", embed=True)

## 9. Video → Text using Gemini 2.5 Flash

Place a short video named `sample_video.mp4` in this folder before running.


In [ ]:
video_path = 'sample_video.mp4'
uploaded_video = gemini_client.files.upload(file=video_path)
active_video = wait_for_gemini_file_active(gemini_client, uploaded_video)
response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=[active_video, 'Summarize this video in 5 bullet points.']
)
print(response.text)


## 10. Final Model List

| Modality | Model | Provider |
|---|---|---|
| Text → Text | gemini-2.5-flash | Google Gemini |
| Text → Image | stabilityai/stable-diffusion-xl-base-1.0 | Hugging Face |
| Image → Text | gemini-2.5-flash | Google Gemini |
| Audio → Text | openai/whisper-large-v3-turbo | Hugging Face |
| Text → Audio | gemini-2.5-flash-preview-tts | Google Gemini |
| Text → Video | SDXL + LTX Video image-to-video | Hugging Face |
| Video → Text | gemini-2.5-flash | Google Gemini |
